In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
from tqdm.notebook import tqdm
from src.data.data_loader import get_dataset

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

Path("results").mkdir(exist_ok=True)

print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}")

In [ ]:
from src.data.data_loader import get_dataset
from src.evaluation.experiments import run_node_classification_experiment

In [ ]:
datasets_to_load = ['cora', 'pubmed', 'facebook', 'twitch_ru', 'lastfm_asia', 'ogbn_arxiv', 'reddit']

graphs = {}
for name in datasets_to_load:
    ds = get_dataset(name, verbose=True)
    graphs[name] = ds

In [ ]:
nc_datasets = ['cora', 'pubmed', 'twitch_ru', 'lastfm_asia', 'ogbn_arxiv']
models = ['logistic','mlp','gcn','graphsage','gat','gatv2',
          'graph_transformer','gin','sgc','jknet','gcnii']

nc_results = []

for ds_name in tqdm(nc_datasets, desc="Datasets"):
    ds = graphs[ds_name]
    for model in tqdm(models, desc=f"Models for {ds_name}", leave=False):
        try:
            res = run_node_classification_experiment(
                ds_name, 
                model=model, 
                use_features=True, 
                use_embeddings=False,
                dimensions=64, 
                n_runs=(2 if ds_name == 'ogbn_arxiv' else (3 if ds_name == 'twitch_ru' else 5)), 
                force_recompute=False,
                balance_loss=(True if ds_name in ['twitch_ru', 'ogbn_arxiv'] else False)
            )
            nc_results.append(res)
            tqdm.write(f"OK {ds_name} {model}: Acc={res['accuracy_mean']:.4f}, MacroF1={res['macro_f1_mean']:.4f}, WeightedF1={res['weighted_f1_mean']:.4f}")
        except Exception as e:
            tqdm.write(f"FAIL {ds_name} {model}: {repr(e)}")

In [ ]:
df_nc = pd.DataFrame(nc_results)
df_nc.to_csv('results/node_classification_results.csv', index=False)
df_nc

In [ ]:
import optuna
best_gnn = {}

for ds_name in nc_datasets:
    ds_res = [r for r in nc_results if r.get('dataset') == ds_name]
    gnn_res = [r for r in ds_res if r.get('model') in ['gcn','graphsage','gat','gatv2','graph_transformer','gin','jknet','gcnii']]
    if gnn_res:
        best = max(gnn_res, key=lambda x: x.get('accuracy_mean', 0))
        best_gnn[ds_name] = best['model']
        print(f"Best GNN for {ds_name}: {best['model']} (Acc = {best.get('accuracy_mean', 0):.4f})")

# Optuna tuning
hyper_results = []

for ds_name, best_model in best_gnn.items():
    print(f"\n🔍 Optuna tuning for {ds_name} → {best_model} (20 trials)")

    def objective(trial):
        hidden_dim = trial.suggest_int("hidden_dim", 32, 256, step=32)
        num_layers = trial.suggest_int("num_layers", 1, 4)
        dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.05)

        res = run_node_classification_experiment(
            ds_name,
            model=best_model,
            use_features=True,
            use_embeddings=False,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            seed=42,
            n_runs=2,
            force_recompute=False,
            balance_loss=(ds_name in ['twitch_ru', 'ogbn_arxiv'])
        )
        return res.get('accuracy_mean', 0.0)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=20, show_progress_bar=True)

    best_params = study.best_params
    best_value = study.best_value

    # Финальный запуск с лучшими параметрами (больше runs)
    final_res = run_node_classification_experiment(
        ds_name,
        model=best_model,
        use_features=True,
        use_embeddings=False,
        hidden_dim=best_params["hidden_dim"],
        num_layers=best_params["num_layers"],
        dropout=best_params["dropout"],
        n_runs=3,
        force_recompute=False,
        balance_loss=(ds_name in ['twitch_ru', 'ogbn_arxiv'])
    )

    hyper_results.append({
        "dataset": ds_name,
        "best_model": best_model,
        "hidden_dim": best_params["hidden_dim"],
        "num_layers": best_params["num_layers"],
        "dropout": best_params["dropout"],
        "best_trial_acc": best_value,
        "final_test_accuracy": final_res.get('accuracy_mean', 0),
        **{k: v for k, v in final_res.items() if k in ["macro_f1_mean", "weighted_f1_mean"]}
    })

    print(f"   Best params: {best_params} | Final Acc = {final_res.get('accuracy_mean', 0):.4f}")